In [ ]:
# Compare Silver article and URL counts with Auto Loader's tracked file count.
from pyspark.sql import functions as F

table_name = '`news-analytics`.silver.articles'
checkpoint_path = '/Volumes/news-analytics/silver/article_ingestion_state/kontrast_articles'
articles = spark.table(table_name)
display(articles.agg(
    F.count('*').alias('articles'),
    F.countDistinct('canonical_url').alias('distinct_urls'),
))
display(spark.sql(f"SELECT COUNT(*) AS tracked_files FROM cloud_files_state('{checkpoint_path}')"))


In [ ]:
# List tracked files that are not the selected source of a Silver article.
unrepresented = spark.sql(f'''
    SELECT CONCAT('s3://', f.path) AS source_path
    FROM cloud_files_state('{checkpoint_path}') AS f
    LEFT ANTI JOIN {table_name} AS a
        ON CONCAT('s3://', f.path) = a.source_path
''')
display(unrepresented.orderBy('source_path'))
